In [1]:
import pandas as pd

tratamiento = pd.read_csv("trat.csv")

In [2]:
accesible2 = pd.read_csv("/u/bernard0/Carlos_2024/Paper/accesible2.csv")

In [9]:
accesible2.columns

Index(['id_grabacion', 'id_clase', 'id_tutor', 'fecha_tentativa_v2',
       'id_estudiante', 'speaker_tutor', 'certeza', 'clases_sin_estudiante',
       'con_otro', 'futuro', 'tiene_futuro', 'speaker_alumno',
       'promedio_estudiantes', 'nuevo_tutor', 'promedio_clase_actual',
       'constant', 'futuro_frecuente', 'tutor_futuro', 'promedio_mejor_futuro',
       'id_grupo', 'asistio', 'speakers_reales', 'categoria'],
      dtype='object')

In [8]:
accesible2

,id_grabacion,id_clase,id_tutor,fecha_tentativa_v2,id_estudiante,speaker_tutor,certeza,clases_sin_estudiante,con_otro,futuro,...,nuevo_tutor,promedio_clase_actual,constant,futuro_frecuente,tutor_futuro,promedio_mejor_futuro,id_grupo,asistio,speakers_reales,categoria
0,127087,241896,54827,2022-08-08,48257,SPEAKER_00,73,"[241921, 241923, 241924, 241929, 241930, 24193...","[np.int64(402147), np.int64(402151), np.int64(...","[402147, 402151, 402152, 402162, 402168, 40217...",...,0.076507,0.026524,1,"[402147, 402151, 402152, 402162, 402168, 40217...",67603,0.142836,6928.0,5,3,asistio > reales
1,127120,241944,54634,2022-08-08,10968,SPEAKER_00,77,[241945],"[np.int64(499402), np.int64(499403), np.int64(...","[499402, 499403, 499408]",...,0.158258,0.203311,1,"[499402, 499403, 499408]",87600,0.495137,6890.0,4,3,asistio > reales
2,127120,241944,54634,2022-08-08,4152,SPEAKER_00,77,"[241954, 241949]",[np.int64(370739)],[370739],...,0.057337,0.203311,1,[370739],54989,0.417449,6890.0,4,3,asistio > reales
3,127401,241104,54818,2022-08-08,45628,SPEAKER_02,62,"[241092, 241095, 241096, 241097, 241098, 24109...","[np.int64(408123), np.int64(408124), np.int64(...","[408123, 408124, 408126, 408129, 408132, 40813...",...,0.059142,0.010240,1,"[496214, 496215, 496216, 496217, 496224, 49622...",87579,0.483952,7023.0,4,2,asistio > reales
4,127415,241080,54818,2022-08-08,22536,SPEAKER_01,116,"[241121, 241122, 241127, 241104, 241106, 24110...","[np.int64(403160), np.int64(403161), np.int64(...","[403160, 403161, 403162, 403163, 403175, 53675...",...,0.058254,0.000079,1,"[403160, 403161, 403162, 403163, 403175]",70187,0.316184,6859.0,5,2,asistio > reales
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12968,345191,650153,92596,2024-04-15,88517,SPEAKER_00,69,[625355],"[np.int64(533188), np.int64(533194), np.int64(...",[],...,0.000000,0.124739,1,[],0,NaN,17069.0,2,3,asistio = reales
12969,345191,650153,92596,2024-04-15,62630,SPEAKER_00,69,[625355],"[np.int64(409217), np.int64(409220), np.int64(...",[],...,0.000000,0.008407,1,[],0,NaN,17069.0,2,3,asistio = reales
12970,350602,700606,93283,2024-05-13,15797,SPEAKER_03,36,"[700769, 700709]","[np.int64(248061), np.int64(248062), np.int64(...",[],...,0.000000,0.099757,1,[],0,NaN,17898.0,1,3,asistio < reales
12971,350709,674799,92011,2024-05-13,55443,SPEAKER_01,47,"[674795, 674663]","[np.int64(307350), np.int64(307354), np.int64(...",[],...,0.000000,0.163058,1,[],0,NaN,17589.0,2,4,asistio < reales


In [4]:
df = accesible2.copy()

df["fecha_tentativa_v2"] = pd.to_datetime(df["fecha_tentativa_v2"], errors="coerce")
df["speaker_alumno"] = pd.to_numeric(df["speaker_alumno"], errors="coerce")
df["speaker_tutor"] = pd.to_numeric(df["speaker_tutor"], errors="coerce")

In [5]:
print(df[["speaker_alumno", "speaker_tutor"]].dtypes)
print(df["speaker_alumno"].head(10))

speaker_alumno    float64
speaker_tutor     float64
dtype: object
0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
7   NaN
8   NaN
9   NaN
Name: speaker_alumno, dtype: float64


In [43]:
import pandas as pd
import numpy as np

def construir_df_cambio_tutor(accesible2, min_visitas_grupo=2):
    df = accesible2.copy()

    # --------------------------
    # 0) Limpieza básica
    # --------------------------
    df["fecha_tentativa_v2"] = pd.to_datetime(df["fecha_tentativa_v2"])
    
    cols_necesarias = [
        "id_grabacion", "id_clase", "id_tutor", "fecha_tentativa_v2",
        "id_estudiante", "promedio_clase_actual"
    ]
    faltantes = [c for c in cols_necesarias if c not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas en accesible2: {faltantes}")

    # Nos quedamos solo con filas válidas
    df = df.dropna(subset=["id_estudiante", "id_tutor", "fecha_tentativa_v2", "promedio_clase_actual"]).copy()

    # --------------------------
    # 1) Buscar fecha de corte óptima
    # --------------------------
    fechas_candidatas = np.sort(df["fecha_tentativa_v2"].dropna().unique())

    resumen_fechas = []

    for fecha_corte in fechas_candidatas:
        df_x = df[df["fecha_tentativa_v2"] < fecha_corte].copy()
        df_y = df[df["fecha_tentativa_v2"] > fecha_corte].copy()

        if df_x.empty or df_y.empty:
            continue

        # Tutores previos por alumno
        prev_tutores = (
            df_x.groupby("id_estudiante")["id_tutor"]
            .apply(lambda s: set(s.dropna().unique()))
            .rename("tutores_previos")
        )

        # Tutores posteriores por alumno
        post_tutores = (
            df_y.groupby("id_estudiante")["id_tutor"]
            .apply(lambda s: set(s.dropna().unique()))
            .rename("tutores_posteriores")
        )

        temp = pd.concat([prev_tutores, post_tutores], axis=1).dropna().reset_index()

        # Contar alumnos que tengan al menos un tutor antes y otro después, y que exista al menos un par distinto x != y
        def tiene_par_distinto(row):
            return any(tx != ty for tx in row["tutores_previos"] for ty in row["tutores_posteriores"])

        temp["valido"] = temp.apply(tiene_par_distinto, axis=1)
        n_validos = temp["valido"].sum()

        resumen_fechas.append({
            "fecha_corte": fecha_corte,
            "n_estudiantes_validos": n_validos
        })

    if not resumen_fechas:
        raise ValueError("No se pudo encontrar ninguna fecha de corte válida.")

    resumen_fechas = pd.DataFrame(resumen_fechas)

    # Fecha que maximiza la cantidad de alumnos válidos
    fecha_optima = resumen_fechas.sort_values(
        ["n_estudiantes_validos", "fecha_corte"],
        ascending=[False, True]
    ).iloc[0]["fecha_corte"]

    print("Fecha de corte óptima:", fecha_optima)
    print("Número de estudiantes válidos:", 
          resumen_fechas.loc[resumen_fechas["fecha_corte"] == fecha_optima, "n_estudiantes_validos"].iloc[0])

    # --------------------------
    # 2) Separar grupos X e Y
    # --------------------------
    df_x = df[df["fecha_tentativa_v2"] < fecha_optima].copy()
    df_y = df[df["fecha_tentativa_v2"] > fecha_optima].copy()

    # --------------------------
    # 3) Agregados del alumno por tutor en cada lado
    #    promedio_grupo_x = promedio del alumno con tutor x antes del corte
    #    promedio_grupo_y = promedio del alumno con tutor y después del corte
    # --------------------------
    # --------------------------
    # 3) Elegir, por estudiante:
    #    - X = último tutor antes o en la fecha de corte
    #    - Y = primer tutor después de la fecha de corte
    #          pero distinto de X; si el primero es igual a X,
    #          tomar el primer tutor distinto de X
    # --------------------------

    # Ojo: aquí X incluye la fecha de corte
    df_x = df[df["fecha_tentativa_v2"] <= fecha_optima].copy()
    df_y = df[df["fecha_tentativa_v2"] > fecha_optima].copy()

    # Última observación antes o en la fecha de corte por estudiante
    ultimo_x = (
        df_x.sort_values(["id_estudiante", "fecha_tentativa_v2", "id_grabacion"])
        .groupby("id_estudiante", as_index=False)
        .tail(1)[["id_estudiante", "id_tutor", "fecha_tentativa_v2"]]
        .rename(columns={"id_tutor": "id_tutor_x"})
    )

    # Primer tutor distinto después de la fecha de corte
    candidato_y = df_y.sort_values(["id_estudiante", "fecha_tentativa_v2", "id_grabacion"]).merge(
        ultimo_x[["id_estudiante", "id_tutor_x"]],
        on="id_estudiante",
        how="inner"
    )

    candidato_y = candidato_y[candidato_y["id_tutor"] != candidato_y["id_tutor_x"]].copy()

    primer_y = (
        candidato_y.groupby("id_estudiante", as_index=False)
        .head(1)[["id_estudiante", "id_tutor", "fecha_tentativa_v2"]]
        .rename(columns={
            "id_tutor": "id_tutor_y",
            "fecha_tentativa_v2": "fecha_tentativa_v2_y"
        })
    )

    # Agregados del alumno usando SOLO los tutores elegidos X y Y

    # X
    alumno_x = (
        ultimo_x[["id_estudiante", "id_tutor_x", "fecha_tentativa_v2"]]
        .merge(
            df_x,
            left_on=["id_estudiante", "id_tutor_x"],
            right_on=["id_estudiante", "id_tutor"],
            how="left",
            suffixes=("", "_df")
        )
    )

    alumno_x = (
        alumno_x.groupby(["id_estudiante", "id_tutor_x", "fecha_tentativa_v2"], as_index=False)
        .agg(
            promedio_grupo_x=("promedio_clase_actual", "mean"),
            n_clases_x=("id_grabacion", "count")
        )
    )

    # Y
    alumno_y = (
        primer_y[["id_estudiante", "id_tutor_y", "fecha_tentativa_v2_y"]]
        .merge(
            df_y,
            left_on=["id_estudiante", "id_tutor_y"],
            right_on=["id_estudiante", "id_tutor"],
            how="left",
            suffixes=("", "_df")
        )
    )

    alumno_y = (
        alumno_y.groupby(["id_estudiante", "id_tutor_y", "fecha_tentativa_v2_y"], as_index=False)
        .agg(
            promedio_grupo_y=("promedio_clase_actual", "mean"),
            n_clases_y=("id_grabacion", "count")
        )
    )

    # --------------------------
    # 4) Crear pares x-y por estudiante con tutores distintos
    # --------------------------
    pares = alumno_x.merge(alumno_y, on="id_estudiante", how="inner")
    pares = pares[pares["id_tutor_x"] != pares["id_tutor_y"]].copy()

    # Excluir grupos donde el alumno fue menos de 2 veces
    pares = pares[
        (pares["n_clases_x"] >= min_visitas_grupo) &
        (pares["n_clases_y"] >= min_visitas_grupo)
    ].copy()

    # --------------------------
    # 5) Promedio general del tutor en X y Y, excluyendo al alumno actual
    #    OJO: aquí se usa promedio por fila/sesión de promedio_clase_actual
    # --------------------------
    # Para poder excluir al alumno actual:
    # promedio_tutor_excl = (suma_total_tutor - suma_alumno_con_tutor) / (n_total_tutor - n_alumno_con_tutor)

    # Totales tutor X
    tutor_x_total = (
        df_x.groupby("id_tutor", as_index=False)
        .agg(
            suma_tutor_x=("promedio_clase_actual", "sum"),
            n_tutor_x=("promedio_clase_actual", "count")
        )
        .rename(columns={"id_tutor": "id_tutor_x"})
    )

    alumno_tutor_x = (
        df_x.groupby(["id_estudiante", "id_tutor"], as_index=False)
        .agg(
            suma_alumno_x=("promedio_clase_actual", "sum"),
            n_alumno_x=("promedio_clase_actual", "count")
        )
        .rename(columns={"id_tutor": "id_tutor_x"})
    )

    # Totales tutor Y
    tutor_y_total = (
        df_y.groupby("id_tutor", as_index=False)
        .agg(
            suma_tutor_y=("promedio_clase_actual", "sum"),
            n_tutor_y=("promedio_clase_actual", "count")
        )
        .rename(columns={"id_tutor": "id_tutor_y"})
    )

    alumno_tutor_y = (
        df_y.groupby(["id_estudiante", "id_tutor"], as_index=False)
        .agg(
            suma_alumno_y=("promedio_clase_actual", "sum"),
            n_alumno_y=("promedio_clase_actual", "count")
        )
        .rename(columns={"id_tutor": "id_tutor_y"})
    )

    # Merge para X
    pares = pares.merge(alumno_tutor_x, on=["id_estudiante", "id_tutor_x"], how="left")
    pares = pares.merge(tutor_x_total, on="id_tutor_x", how="left")

    den_x = pares["n_tutor_x"] - pares["n_alumno_x"]
    num_x = pares["suma_tutor_x"] - pares["suma_alumno_x"]

    pares["promedio_tutor_grupo_x"] = np.where(
        den_x > 0,
        num_x / den_x,
        np.nan
    )

    # Merge para Y
    pares = pares.merge(alumno_tutor_y, on=["id_estudiante", "id_tutor_y"], how="left")
    pares = pares.merge(tutor_y_total, on="id_tutor_y", how="left")

    den_y = pares["n_tutor_y"] - pares["n_alumno_y"]
    num_y = pares["suma_tutor_y"] - pares["suma_alumno_y"]

    pares["promedio_tutor_grupo_y"] = np.where(
        den_y > 0,
        num_y / den_y,
        np.nan
    )

    # --------------------------
    # 6) id_grabacion
    #    Como ahora cada fila representa un par (alumno, tutor_x, tutor_y),
    #    no hay un único id_grabacion natural para todo el par.
    #    Aquí usaré el último id_grabacion del grupo X como identificador.
    #    Si prefieres otro criterio, lo cambiamos.
    # --------------------------
    ultimo_id_x = (
        df_x.sort_values("fecha_tentativa_v2")
        .groupby(["id_estudiante", "id_tutor"], as_index=False)
        .last()[["id_estudiante", "id_tutor", "id_grabacion"]]
        .rename(columns={"id_tutor": "id_tutor_x"})
    )

    pares = pares.merge(ultimo_id_x, on=["id_estudiante", "id_tutor_x"], how="left")

    # --------------------------
    # 7) Dar formato final
    # --------------------------
    #df_final = pares.rename(columns={"id_tutor_x": "id_tutor"}).copy()
    df_final = pares.rename(columns={}).copy()

    df_final = df_final[
        [
            "id_grabacion",
            "id_estudiante",
            "id_tutor_x",
            "id_tutor_y",
            "fecha_tentativa_v2",
            "fecha_tentativa_v2_y",
            "promedio_grupo_x",
            "promedio_grupo_y",
            "promedio_tutor_grupo_x",
            "promedio_tutor_grupo_y"
        ]
    ].copy()

    # Si quieres exactamente el nombre con acento:
    df_final = df_final.rename(columns={"id_grabacion": "id_grabación"})

    return df_final, fecha_optima, resumen_fechas

In [44]:
df_nuevo, fecha_optima, resumen_fechas = construir_df_cambio_tutor(accesible2, min_visitas_grupo=2)

Fecha de corte óptima: 2023-04-23 00:00:00
Número de estudiantes válidos: 1009


In [46]:
df_nuevo.to_csv("/u/bernard0/Carlos_2024/Paper/datos_limpios2.csv" , index = False)

In [26]:
np.corrcoef(
    *df_nuevo[["promedio_tutor_grupo_y", "promedio_tutor_grupo_x"]]
    .dropna()
    .to_numpy()
    .T
)

array([[1.        , 0.04352572],
       [0.04352572, 1.        ]])

In [27]:
np.corrcoef(
    *df_nuevo[["promedio_grupo_x", "promedio_tutor_grupo_y"]]
    .dropna()
    .to_numpy()
    .T
)

array([[1.        , 0.01777934],
       [0.01777934, 1.        ]])

In [40]:
df_nuevo.to_csv("/u/bernard0/Carlos_2024/Paper/datos_limpios.csv" , index = False)

In [28]:
import statsmodels.api as sm

df_tmp = df_nuevo[["promedio_grupo_x", "promedio_tutor_grupo_y"]].dropna()

X = sm.add_constant(df_tmp["promedio_grupo_x"])
y = df_tmp["promedio_tutor_grupo_y"]

model = sm.OLS(y, X).fit()

print(model.summary())

                              OLS Regression Results                              
Dep. Variable:     promedio_tutor_grupo_y   R-squared:                       0.000
Model:                                OLS   Adj. R-squared:                 -0.002
Method:                     Least Squares   F-statistic:                    0.1347
Date:                    Fri, 06 Mar 2026   Prob (F-statistic):              0.714
Time:                            20:25:57   Log-Likelihood:                 418.71
No. Observations:                     428   AIC:                            -833.4
Df Residuals:                         426   BIC:                            -825.3
Df Model:                               1                                         
Covariance Type:                nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------


In [29]:
import statsmodels.api as sm

df_tmp = df_nuevo[["promedio_grupo_x", "promedio_grupo_y"]].dropna()

X = sm.add_constant(df_tmp["promedio_grupo_x"])
y = df_tmp["promedio_grupo_y"]

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       promedio_grupo_y   R-squared:                       0.040
Model:                            OLS   Adj. R-squared:                  0.038
Method:                 Least Squares   F-statistic:                     23.78
Date:                Fri, 06 Mar 2026   Prob (F-statistic):           1.41e-06
Time:                        20:26:55   Log-Likelihood:                 633.37
No. Observations:                 573   AIC:                            -1263.
Df Residuals:                     571   BIC:                            -1254.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.0887      0.006  

In [32]:
import statsmodels.api as sm

df_tmp = df_nuevo[["promedio_grupo_y", "promedio_tutor_grupo_y"]].dropna()

X = sm.add_constant(df_tmp["promedio_grupo_y"])
y = df_tmp["promedio_tutor_grupo_y"]

model = sm.OLS(y, X).fit()

print(model.summary())

                              OLS Regression Results                              
Dep. Variable:     promedio_tutor_grupo_y   R-squared:                       0.165
Model:                                OLS   Adj. R-squared:                  0.163
Method:                     Least Squares   F-statistic:                     84.17
Date:                    Fri, 06 Mar 2026   Prob (F-statistic):           1.97e-18
Time:                            20:27:51   Log-Likelihood:                 457.22
No. Observations:                     428   AIC:                            -910.4
Df Residuals:                         426   BIC:                            -902.3
Df Model:                               1                                         
Covariance Type:                nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------


In [36]:
import statsmodels.api as sm

df_tmp = df_nuevo[["promedio_grupo_y","promedio_grupo_x", "promedio_tutor_grupo_y"]].dropna()

X = sm.add_constant(df_tmp[["promedio_tutor_grupo_y", "promedio_grupo_x"]])
y = df_tmp["promedio_grupo_y"]

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       promedio_grupo_y   R-squared:                       0.205
Model:                            OLS   Adj. R-squared:                  0.202
Method:                 Least Squares   F-statistic:                     54.96
Date:                Fri, 06 Mar 2026   Prob (F-statistic):           5.90e-22
Time:                        20:29:56   Log-Likelihood:                 496.14
No. Observations:                 428   AIC:                            -986.3
Df Residuals:                     425   BIC:                            -974.1
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      0

In [38]:
import statsmodels.api as sm

df_tmp = df_nuevo[["promedio_grupo_y","promedio_tutor_grupo_x", "promedio_tutor_grupo_y"]].dropna()

X = sm.add_constant(df_tmp[["promedio_tutor_grupo_y", "promedio_tutor_grupo_x"]])
y = df_tmp["promedio_grupo_y"]

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       promedio_grupo_y   R-squared:                       0.260
Model:                            OLS   Adj. R-squared:                  0.255
Method:                 Least Squares   F-statistic:                     52.96
Date:                Fri, 06 Mar 2026   Prob (F-statistic):           1.93e-20
Time:                        20:30:32   Log-Likelihood:                 363.60
No. Observations:                 305   AIC:                            -721.2
Df Residuals:                     302   BIC:                            -710.0
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      0

In [3]:
tratamiento

,Unnamed: 0,id_estudiante,detalles_tratamiento,asignacion_tratamiento,id_estrato,vigente,id_estudiante_ronda,finalizacion_tratamiento,fecha_asignacion_tutor,fecha_TH_end,...,id_encuesta_final,historial,id_tarea,op_end_asignada,th_end_asignada,id_op,id_tarea_op,excepcion,detalles_operacion_tratamiento,id_tratamiento
0,0,53973,{'emocional': 0},2022-08-08 14:04:33,1.0,0,1,2023-08-16 08:07:47,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,{'emocional': 0},1.0
1,1,54283,{'emocional': 1},2022-08-08 15:11:25,2.0,0,2,2023-02-21 19:53:16,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,{'emocional': 1},2.0
2,2,45667,{'emocional': 0},2022-08-08 15:50:56,3.0,0,3,2023-08-16 08:07:47,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,{'emocional': 0},1.0
3,3,55925,{'emocional': 1},2022-08-08 16:14:32,4.0,0,4,2023-02-21 19:53:16,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,{'emocional': 1},2.0
4,4,45511,{'emocional': 0},2022-08-08 16:17:46,2.0,0,5,2023-02-21 19:53:16,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,{'emocional': 0},1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81610,81610,110862,"{'version': 'otonyo2024', 'actividades': 'pre_...",2026-03-02 15:19:08,5.0,1,95509,NaN,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,27.0
81611,81611,110863,"{'version': 'otonyo2024', 'actividades': 'pre_...",2026-03-02 15:19:08,5.0,1,95510,NaN,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,27.0
81612,81612,111202,"{'version': 'otonyo2024', 'actividades': 'pre_...",2026-03-02 16:20:59,48.0,1,95511,NaN,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,27.0
81613,81613,104697,"{'version': 'otonyo2024', 'actividades': 'pre_...",2026-03-02 17:56:03,3.0,1,95512,NaN,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,27.0


In [5]:
import numpy as np
import pandas as pd

# -----------------------------
# 0) Load sample
# -----------------------------
df = pd.read_csv("regression_sample_for_balance.csv")

# Ensure terciles exist (robust to ties)
if "tutor_y_tercile" not in df.columns or df["tutor_y_tercile"].isna().all():
    try:
        df["tutor_y_tercile"] = pd.qcut(
            df["promedios_tutor_y"], q=3, labels=["Low", "Medium", "High"]
        )
    except ValueError:
        df["tutor_y_tercile"] = pd.qcut(
            df["promedios_tutor_y"].rank(method="first"),
            q=3, labels=["Low", "Medium", "High"]
        )

# Make sure ordering is consistent
df["tutor_y_tercile"] = pd.Categorical(
    df["tutor_y_tercile"], categories=["Low", "Medium", "High"], ordered=True
)

# -----------------------------
# 1) Choose covariates for balance
#    (use pre-treatment covariates; add demographics later after merging)
# -----------------------------
balance_vars = [
    "participacion_x",
    "participacion_x_count",
    "promedios_tutor_x",
    "promedios_tutor_y",  # (optional) include since it's the stratifier; often left out of balance rows
]

# keep only those that exist
balance_vars = [v for v in balance_vars if v in df.columns]

# If you don't want the stratifier to appear as a row, uncomment:
# balance_vars = [v for v in balance_vars if v != "promedios_tutor_y"]

# -----------------------------
# 2) Helper: SMD Low vs High
# -----------------------------
def smd_low_high(data: pd.DataFrame, var: str) -> float:
    low = data.loc[data["tutor_y_tercile"] == "Low", var].dropna()
    high = data.loc[data["tutor_y_tercile"] == "High", var].dropna()
    if len(low) < 2 or len(high) < 2:
        return np.nan
    pooled = np.sqrt((low.var(ddof=1) + high.var(ddof=1)) / 2)
    if pooled == 0 or np.isnan(pooled):
        return np.nan
    return (high.mean() - low.mean()) / pooled  # High - Low

# -----------------------------
# 3) Build balance table
# -----------------------------
group_means = df.groupby("tutor_y_tercile")[balance_vars].mean().T
group_sds   = df.groupby("tutor_y_tercile")[balance_vars].std().T
group_ns    = df.groupby("tutor_y_tercile").size()

# SMDs
smds = pd.Series({v: smd_low_high(df, v) for v in balance_vars}, name="SMD (High-Low)")

# Combine: show mean (sd) by tercile + SMD
def fmt_mean_sd(mean, sd):
    if pd.isna(mean):
        return ""
    if pd.isna(sd):
        return f"{mean:.3f}"
    return f"{mean:.3f} ({sd:.3f})"

balance_tbl = pd.DataFrame(index=balance_vars)

for g in ["Low", "Medium", "High"]:
    balance_tbl[g] = [
        fmt_mean_sd(group_means.loc[v, g], group_sds.loc[v, g]) for v in balance_vars
    ]

balance_tbl["SMD (High-Low)"] = [smds.loc[v] for v in balance_vars]
balance_tbl["SMD (High-Low)"] = balance_tbl["SMD (High-Low)"].map(
    lambda x: "" if pd.isna(x) else f"{x:.3f}"
)

# Add N row at the top
n_row = pd.DataFrame(
    {
        "Low": [f"{int(group_ns.get('Low', 0))}"],
        "Medium": [f"{int(group_ns.get('Medium', 0))}"],
        "High": [f"{int(group_ns.get('High', 0))}"],
        "SMD (High-Low)": [""],
    },
    index=["N"]
)

balance_tbl = pd.concat([n_row, balance_tbl], axis=0)

balance_tbl

/tmp/ipykernel_3287395/3386840179.py:59: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_means = df.groupby("tutor_y_tercile")[balance_vars].mean().T
/tmp/ipykernel_3287395/3386840179.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_sds   = df.groupby("tutor_y_tercile")[balance_vars].std().T
/tmp/ipykernel_3287395/3386840179.py:61: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_ns    = df.groupby("tuto

,Low,Medium,High,SMD (High-Low)
N,56,56,56,
participacion_x,0.106 (0.050),0.102 (0.060),0.134 (0.075),0.439
participacion_x_count,8.893 (2.890),9.429 (3.861),9.607 (3.681),0.216


In [6]:
df.columns

Index(['id_estudiante', 'id_grupo_x', 'id_grupo_y', 'id_tutor_x', 'id_tutor_y',
       'fecha_x', 'fecha_y', 'correcta', 'participacion_x', 'participacion_y',
       'participacion_x_count', 'participacion_y_count',
       'promedios_tutor_x_grupo', 'promedios_tutor_y_grupo',
       'tutor_y_tercile'],
      dtype='object')

In [7]:
clases = pd.read_csv("/u/bernard0/Carlos_2024/clases.csv")
asistencia = pd.read_csv("/u/bernard0/Carlos_2024/asistencia.csv")
grabaciones = pd.read_csv("/u/bernard0/Carlos_2024/grabaciones.csv")
estudiantes= pd.read_csv("/u/bernard0/Carlos_2024/estudiantes.csv")

/tmp/ipykernel_3287395/1748657083.py:3: DtypeWarning: Columns (4,10,12,14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  grabaciones = pd.read_csv("/u/bernard0/Carlos_2024/grabaciones.csv")
/tmp/ipykernel_3287395/1748657083.py:4: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  estudiantes= pd.read_csv("/u/bernard0/Carlos_2024/estudiantes.csv")


In [8]:
# 1) Cargar tablas
clases = pd.read_csv("/u/bernard0/Carlos_2024/clases.csv")
asistencia = pd.read_csv("/u/bernard0/Carlos_2024/asistencia.csv")

# 2) Usar SOLO fecha_tentativa
clases["fecha_clase"] = pd.to_datetime(clases["fecha_tentativa"], errors="coerce")

# 3) Construir tabla de eventos asistidos
events = asistencia.merge(
    clases[["id_clase", "id_tutor", "fecha_clase"]],
    on="id_clase",
    how="inner"
)

events = events[
    (events["asistio"] == 1) &
    (~events["fecha_clase"].isna())
].copy()

events["id_tutor"] = events["id_tutor"].astype(int)

# 4) Para cada estudiante, obtener PRIMERA sesión con tutor Y
events = events.merge(
    df[["id_estudiante", "id_tutor_y"]],
    on="id_estudiante",
    how="inner"
)

events_y = events[events["id_tutor"] == events["id_tutor_y"]]

fecha_y_correcta = (
    events_y
    .groupby("id_estudiante")["fecha_clase"]
    .min()
    .rename("fecha_y_correcta")
)

# 5) Reemplazar fecha_y
df = df.merge(fecha_y_correcta, on="id_estudiante", how="left")
df["fecha_y"] = df["fecha_y_correcta"]
df = df.drop(columns=["fecha_y_correcta"])

In [9]:
clases.prefijo

0           Prueba1
1           Prueba1
2           Prueba1
3           Prueba1
4           Prueba1
            ...    
953346    EDU21299N
953347    EDU21299N
953348    EDU21299N
953349    EDU21299N
953350    EDU21299N
Name: prefijo, Length: 953351, dtype: object

In [10]:
print(clases.columns , asistencia.columns , grabaciones.columns , estudiantes.columns)

Index(['id_clase', 'id_tutor', 'prefijo', 'editable', 'nombre', 'reposicion',
       'clase_asignada', 'visible', 'vacaciones', 'fecha_tentativa', 'alpha',
       'id_grupo', 'fecha_tentativa_v2', 'fecha_clase'],
      dtype='object') Index(['id_clase', 'id_estudiante', 'asistio', 'notas', 'razon_falta',
       'justificacion'],
      dtype='object') Index(['id_grabacion', 'id_clase', 'nombre_original', 'exito', 'id_tarea',
       'hash', 'actividad', 'mfcc', 'whisper', 'prioridad', 'en_drive',
       'trabajando', 'host_name', 'id_entrenamiento', 'whisper_ass_file_id',
       'mfcc_file_id', 'rmss_file_id'],
      dtype='object') Index(['id_estudiante', 'apellido', 'nombre', 'edad', 'telefono', 'alias',
       'instituto', 'grado', 'rid', 'alt_cel', 'estado', 'tratamiento',
       'repetido', 'linea_atencion', 'maxgrupos', 'dar_baja',
       'tel_mas_confiable', 'id_docente', 'treshoras', 'suspected_max_grups',
       'prioridad', 'no_responde', 'mas_alumnos', 'carrera', 'genero',
   

In [11]:
clases.fecha_tentativa_v2

0         1970-01-01
1         1970-01-01
2         1970-01-01
3         1970-01-01
4         1970-01-01
             ...    
953346    2025-01-20
953347    2025-01-27
953348    2025-01-27
953349    2025-01-27
953350    2025-01-27
Name: fecha_tentativa_v2, Length: 953351, dtype: object

In [13]:
import pandas as pd
import numpy as np

# ----------------------------------
# 1) Load regression sample
# ----------------------------------
df = pd.read_csv("regression_sample_for_balance.csv")

# ----------------------------------
# 2) Load raw data
# ----------------------------------
clases = pd.read_csv("/u/bernard0/Carlos_2024/clases.csv")
asistencia = pd.read_csv("/u/bernard0/Carlos_2024/asistencia.csv")
estudiantes = pd.read_csv("/u/bernard0/Carlos_2024/estudiantes.csv")

import pandas as pd
import numpy as np


# 2) Usar SOLO fecha_tentativa
clases["fecha_clase"] = pd.to_datetime(clases["fecha_tentativa_v2"], errors="coerce")

# 3) Construir tabla de eventos asistidos
events = asistencia.merge(
    clases[["id_clase", "id_tutor", "fecha_clase"]],
    on="id_clase",
    how="inner"
)

events = events[
    (events["asistio"] == 1) &
    (~events["fecha_clase"].isna())
].copy()

events["id_tutor"] = events["id_tutor"].astype(int)

# 4) Para cada estudiante, obtener PRIMERA sesión con tutor Y
events = events.merge(
    df[["id_estudiante", "id_tutor_y"]],
    on="id_estudiante",
    how="inner"
)

events_y = events[events["id_tutor"] == events["id_tutor_y"]]

fecha_y_correcta = (
    events_y
    .groupby("id_estudiante")["fecha_clase"]
    .min()
    .rename("fecha_y_correcta")
)

# 5) Reemplazar fecha_y
df = df.merge(fecha_y_correcta, on="id_estudiante", how="left")
df["fecha_y"] = df["fecha_y_correcta"]
df = df.drop(columns=["fecha_y_correcta"])

# df = pd.read_csv("regression_sample_for_balance.csv")
df["fecha_y"] = pd.to_datetime(df["fecha_y"], errors="coerce")

clases = pd.read_csv("/u/bernard0/Carlos_2024/clases.csv")
asistencia = pd.read_csv("/u/bernard0/Carlos_2024/asistencia.csv")

# 1) Fecha de la clase (usa v2 si estÃ¡, si no fallback)
clases["fecha_clase"] = clases["fecha_tentativa_v2"] #.fillna(clases["fecha_tentativa"])
clases["fecha_clase"] = pd.to_datetime(clases["fecha_clase"], errors="coerce")

# 2) Asistencia + tutor + fecha
events = asistencia.merge(
    clases[["id_clase", "id_tutor", "fecha_clase"]],
    on="id_clase",
    how="left"
)

# quedarnos con asistencia real
events = events[(events["asistio"] == 1) & (~events["id_tutor"].isna()) & (~events["fecha_clase"].isna())].copy()
events["id_tutor"] = events["id_tutor"].astype(int)

# 3) Traer fecha_y por estudiante y filtrar PRE-Y
events = events.merge(df[["id_estudiante", "fecha_y"]], on="id_estudiante", how="inner")

events_preY = events[events["fecha_clase"] < events["fecha_y"]].copy()

# 4) total sessions pre-Y
total_preY = events_preY.groupby("id_estudiante").size().rename("total_sessions_preY")

df = df.merge(total_preY, on="id_estudiante", how="left")
df["total_sessions_preY"] = df["total_sessions_preY"].fillna(0).astype(int)

# 5) sessions with X pre-Y
#    Hacemos merge con tutor_x por estudiante y contamos
events_preY = events_preY.merge(df[["id_estudiante", "id_tutor_x", "id_tutor_y"]], on="id_estudiante", how="left")

x_preY = (
    events_preY[events_preY["id_tutor"] == events_preY["id_tutor_x"]]
    .groupby("id_estudiante").size()
    .rename("sessions_with_X_preY")
)

y_preY = (
    events_preY[events_preY["id_tutor"] == events_preY["id_tutor_y"]]
    .groupby("id_estudiante").size()
    .rename("sessions_with_Y_preY")
)

df = df.merge(x_preY, on="id_estudiante", how="left")
df = df.merge(y_preY, on="id_estudiante", how="left")

df["sessions_with_X_preY"] = df["sessions_with_X_preY"].fillna(0).astype(int)
df["sessions_with_Y_preY"] = df["sessions_with_Y_preY"].fillna(0).astype(int)

# Sanity check: idealmente sessions_with_Y_preY == 0
print("Sessions with Y pre-Y (should be mostly 0):")
print(df["sessions_with_Y_preY"].value_counts().head(10))

df.head()

# ----------------------------------
# 3) Merge demographics
# ----------------------------------
demo_vars = [
    "id_estudiante",
    "edad",
    "genero",
    "grado",
    "tratamiento",
    "instituto",
    "campus",
]

demo = estudiantes[demo_vars].copy()

df = df.merge(demo, on="id_estudiante", how="left")

# ----------------------------------
# 4) Build attendance measures
# ----------------------------------

# Merge asistencia with tutor
asist = asistencia.merge(
    clases[["id_clase", "id_tutor"]],
    on="id_clase",
    how="left"
)

asist = asist[asist["asistio"] == 1]

# Total sessions attended
total_sessions = asist.groupby("id_estudiante").size().rename("total_sessions")

df = df.merge(total_sessions, on="id_estudiante", how="left")

# Sessions with tutor X
def count_sessions(student, tutor):
    return len(
        asist[(asist["id_estudiante"] == student) &
              (asist["id_tutor"] == tutor)]
    )

df["sessions_with_X"] = df.apply(
    lambda x: count_sessions(x.id_estudiante, x.id_tutor_x),
    axis=1
)

df["sessions_with_Y"] = df.apply(
    lambda x: count_sessions(x.id_estudiante, x.id_tutor_y),
    axis=1
)

# ----------------------------------
# 5) Ensure terciles exist
# ----------------------------------
if "tutor_y_tercile" not in df.columns:
    df["tutor_y_tercile"] = pd.qcut(
        df["promedios_tutor_y"].rank(method="first"),
        q=3,
        labels=["Low", "Medium", "High"]
    )

df["tutor_y_tercile"] = pd.Categorical(
    df["tutor_y_tercile"],
    categories=["Low", "Medium", "High"],
    ordered=True
)

# ----------------------------------
# 6) Variables for balance
# ----------------------------------

balance_vars = [
    # Pre-treatment participation
    "participacion_x",
    "participacion_x_count",
    "promedios_tutor_x",

    # Demographics
    "edad",
    "grado",
    "tratamiento",   

    # Attendance intensity
    "total_sessions",
    "sessions_with_X",
    
    "sessions_with_X_preY",
    "sessions_with_Y_preY",
    "total_sessions_preY"
    
]

balance_vars = [v for v in balance_vars if v in df.columns]

# Convert gender to numeric dummy if needed
if "genero" in df.columns:
    df["genero_female"] = (df["genero"] == "F").astype(int)
    balance_vars.append("genero_female")

# ----------------------------------
# 7) Compute balance table
# ----------------------------------

def smd_low_high(data, var):
    low = data[data["tutor_y_tercile"] == "Low"][var].dropna()
    high = data[data["tutor_y_tercile"] == "High"][var].dropna()
    pooled = np.sqrt((low.var(ddof=1) + high.var(ddof=1)) / 2)
    if pooled == 0 or np.isnan(pooled):
        return np.nan
    return (high.mean() - low.mean()) / pooled

means = df.groupby("tutor_y_tercile")[balance_vars].mean().T
sds = df.groupby("tutor_y_tercile")[balance_vars].std().T
Ns = df.groupby("tutor_y_tercile").size()

balance_table = pd.DataFrame(index=balance_vars)

for g in ["Low", "Medium", "High"]:
    balance_table[g] = [
        f"{means.loc[v,g]:.3f} ({sds.loc[v,g]:.3f})"
        for v in balance_vars
    ]

balance_table["SMD (High-Low)"] = [
    f"{smd_low_high(df,v):.3f}" for v in balance_vars
]

# Add N row
n_row = pd.DataFrame(
    {
        "Low":[Ns["Low"]],
        "Medium":[Ns["Medium"]],
        "High":[Ns["High"]],
        "SMD (High-Low)":[""]
    },
    index=["N"]
)

balance_table = pd.concat([n_row, balance_table])

balance_table

/tmp/ipykernel_3287395/1516591559.py:14: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  estudiantes = pd.read_csv("/u/bernard0/Carlos_2024/estudiantes.csv")


Sessions with Y pre-Y (should be mostly 0):
sessions_with_Y_preY
0    168
Name: count, dtype: int64


/tmp/ipykernel_3287395/1516591559.py:230: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  means = df.groupby("tutor_y_tercile")[balance_vars].mean().T
/tmp/ipykernel_3287395/1516591559.py:231: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sds = df.groupby("tutor_y_tercile")[balance_vars].std().T
/tmp/ipykernel_3287395/1516591559.py:232: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  Ns = df.groupby("tutor_y_tercile").size()

,Low,Medium,High,SMD (High-Low)
N,56,56,56,
participacion_x,0.106 (0.050),0.102 (0.060),0.134 (0.075),0.439
participacion_x_count,8.893 (2.890),9.429 (3.861),9.607 (3.681),0.216
edad,10.000 (0.000),10.000 (0.000),10.000 (0.000),nan
grado,7.268 (1.446),7.518 (1.561),7.536 (1.464),0.184
tratamiento,nan (nan),nan (nan),nan (nan),nan
total_sessions,94.946 (34.046),88.839 (31.083),103.732 (35.600),0.252
sessions_with_X,22.536 (6.255),22.357 (6.471),24.054 (6.431),0.239
sessions_with_X_preY,52.732 (60.763),38.089 (34.841),73.429 (80.032),0.291
sessions_with_Y_preY,0.000 (0.000),0.000 (0.000),0.000 (0.000),nan


In [14]:
grupos_high=df[df.tutor_y_tercile=="High"].id_grupo_x+df[df.tutor_y_tercile=="High"].id_grupo_y

clases[clases.id_grupo.isin(grupos_high)].prefijo

624374    EDU17290N
624375    EDU17290N
624376    EDU17290N
624377    EDU17290N
624378    EDU17290N
            ...    
950727    EDU21306N
950728    EDU21306N
950729    EDU21306N
950730    EDU21306N
950731    EDU21306N
Name: prefijo, Length: 1166, dtype: object

In [29]:
# INVESTIGAR

In [15]:
import pandas as pd
import numpy as np

df = pd.read_csv("regression_sample_for_balance.csv")
clases = pd.read_csv("/u/bernard0/Carlos_2024/clases.csv")
asistencia = pd.read_csv("/u/bernard0/Carlos_2024/asistencia.csv")

# 1) SOLO fecha_tentativa_v2
clases["fecha_clase"] = pd.to_datetime(
    clases["fecha_tentativa_v2"],
    format="%Y-%m-%d",
    errors="coerce"
)

# 2) Events (asistencias reales) con tutor y fecha válida
events = (asistencia
    .merge(clases[["id_clase", "id_tutor", "fecha_clase"]], on="id_clase", how="left")
    .query("asistio == 1")
    .dropna(subset=["id_tutor", "fecha_clase"])
    .copy()
)
events["id_tutor"] = events["id_tutor"].astype(int)

# 3) Traer tutores X/Y al nivel event
df["id_tutor_y"] = pd.to_numeric(df["id_tutor_y"], errors="coerce").astype("Int64")
df["id_tutor_x"] = pd.to_numeric(df["id_tutor_x"], errors="coerce").astype("Int64")

events_xy = events.merge(
    df[["id_estudiante", "id_tutor_y", "id_tutor_x"]],
    on="id_estudiante",
    how="inner"
)

# 4) fecha_y = primera sesión con Y (usando SOLO v2)
events_y = events_xy[events_xy["id_tutor"].eq(events_xy["id_tutor_y"])]

fecha_y = (events_y
    .groupby("id_estudiante")["fecha_clase"]
    .min()
    .rename("fecha_y")
)

# guardar en df
df = df.drop(columns=["fecha_y"], errors="ignore").merge(fecha_y, on="id_estudiante", how="left")

# 5) IMPORTANTe: meter fecha_y a nivel event para poder filtrar preY
events_xy = events_xy.merge(df[["id_estudiante", "fecha_y"]], on="id_estudiante", how="left")

# 6) Pre-Y (solo donde fecha_y existe)
events_preY = events_xy[
    events_xy["fecha_y"].notna() &
    (events_xy["fecha_clase"] < events_xy["fecha_y"])
].copy()

# 7) Conteos pre-Y
total_preY = events_preY.groupby("id_estudiante").size().rename("total_sessions_preY")

x_preY = (events_preY[events_preY["id_tutor"].eq(events_preY["id_tutor_x"])]
          .groupby("id_estudiante").size().rename("sessions_with_X_preY"))

y_preY = (events_preY[events_preY["id_tutor"].eq(events_preY["id_tutor_y"])]
          .groupby("id_estudiante").size().rename("sessions_with_Y_preY"))

df = df.merge(total_preY, on="id_estudiante", how="left")
df = df.merge(x_preY, on="id_estudiante", how="left")
df = df.merge(y_preY, on="id_estudiante", how="left")

df["total_sessions_preY"] = df["total_sessions_preY"].fillna(0).astype(int)
df["sessions_with_X_preY"] = df["sessions_with_X_preY"].fillna(0).astype(int)
df["sessions_with_Y_preY"] = df["sessions_with_Y_preY"].fillna(0).astype(int)

print("Sessions with Y pre-Y (should be 0):")
print(df["sessions_with_Y_preY"].value_counts().head(10))

print("Share fecha_y NaT:", df["fecha_y"].isna().mean())
print("events_xy rows:", len(events_xy), "| events_preY rows:", len(events_preY))

Sessions with Y pre-Y (should be 0):
sessions_with_Y_preY
0    168
Name: count, dtype: int64
Share fecha_y NaT: 0.0
events_xy rows: 26641 | events_preY rows: 12186


In [16]:
# ----------------------------------
# (SIGUE DESPUÉS DE TU CÓDIGO CORREGIDO)
# 3) Merge demographics
# ----------------------------------
demo_vars = [
    "id_estudiante",
    "edad",
    "genero",
    "grado",
    "tratamiento",
    "instituto",
    "campus",
]

demo = estudiantes[demo_vars].copy()
df = df.merge(demo, on="id_estudiante", how="left")

# ----------------------------------
# 4) Build attendance measures (TOTAL, X, Y)
#    OJO: aquí también usamos SOLO fecha_tentativa_v2
# ----------------------------------
clases_tmp = pd.read_csv("/u/bernard0/Carlos_2024/clases.csv")
asistencia_tmp = pd.read_csv("/u/bernard0/Carlos_2024/asistencia.csv")

clases_tmp["fecha_clase"] = pd.to_datetime(
    clases_tmp["fecha_tentativa_v2"],
    format="%Y-%m-%d",
    errors="coerce"
)

asist = (asistencia_tmp
    .merge(clases_tmp[["id_clase", "id_tutor", "fecha_clase"]], on="id_clase", how="left")
    .query("asistio == 1")
    .dropna(subset=["id_tutor", "fecha_clase"])
    .copy()
)
asist["id_tutor"] = asist["id_tutor"].astype(int)

# Total sessions attended
total_sessions = asist.groupby("id_estudiante").size().rename("total_sessions")
df = df.merge(total_sessions, on="id_estudiante", how="left")
df["total_sessions"] = df["total_sessions"].fillna(0).astype(int)

# Sessions with tutor X/Y (vectorizado, sin apply)
sx = asist.merge(df[["id_estudiante", "id_tutor_x"]], on="id_estudiante", how="inner")
sessions_with_X = (sx[sx["id_tutor"].eq(sx["id_tutor_x"])]
                   .groupby("id_estudiante").size().rename("sessions_with_X"))

sy = asist.merge(df[["id_estudiante", "id_tutor_y"]], on="id_estudiante", how="inner")
sessions_with_Y = (sy[sy["id_tutor"].eq(sy["id_tutor_y"])]
                   .groupby("id_estudiante").size().rename("sessions_with_Y"))

df = df.merge(sessions_with_X, on="id_estudiante", how="left")
df = df.merge(sessions_with_Y, on="id_estudiante", how="left")
df["sessions_with_X"] = df["sessions_with_X"].fillna(0).astype(int)
df["sessions_with_Y"] = df["sessions_with_Y"].fillna(0).astype(int)

# ----------------------------------
# 5) Ensure terciles exist
# ----------------------------------
if "tutor_y_tercile" not in df.columns:
    df["tutor_y_tercile"] = pd.qcut(
        df["promedios_tutor_y"].rank(method="first"),
        q=3,
        labels=["Low", "Medium", "High"]
    )

df["tutor_y_tercile"] = pd.Categorical(
    df["tutor_y_tercile"],
    categories=["Low", "Medium", "High"],
    ordered=True
)

# ----------------------------------
# 6) Variables for balance
# ----------------------------------
balance_vars = [
    # Pre-treatment participation
    "participacion_x",
    "participacion_x_count",
    "promedios_tutor_x",

    # Demographics
    "edad",
    "grado",
    "tratamiento",

    # Attendance intensity
    "total_sessions",
    "sessions_with_X",

    "sessions_with_X_preY",
    "sessions_with_Y_preY",
    "total_sessions_preY",
]

balance_vars = [v for v in balance_vars if v in df.columns]

# Convert gender to numeric dummy if needed
if "genero" in df.columns and "genero_female" not in df.columns:
    df["genero_female"] = (df["genero"] == "F").astype(int)
if "genero_female" in df.columns and "genero_female" not in balance_vars:
    balance_vars.append("genero_female")

# ----------------------------------
# 7) Compute balance table (igual que tu original)
# ----------------------------------
def smd_low_high(data, var):
    low = data.loc[data["tutor_y_tercile"] == "Low", var].dropna()
    high = data.loc[data["tutor_y_tercile"] == "High", var].dropna()
    pooled = np.sqrt((low.var(ddof=1) + high.var(ddof=1)) / 2)
    if pooled == 0 or np.isnan(pooled):
        return np.nan
    return (high.mean() - low.mean()) / pooled

means = df.groupby("tutor_y_tercile")[balance_vars].mean().T
sds = df.groupby("tutor_y_tercile")[balance_vars].std().T
Ns = df.groupby("tutor_y_tercile").size()

balance_table = pd.DataFrame(index=balance_vars)

for g in ["Low", "Medium", "High"]:
    balance_table[g] = [
        f"{means.loc[v, g]:.3f} ({sds.loc[v, g]:.3f})"
        for v in balance_vars
    ]

balance_table["SMD (High-Low)"] = [f"{smd_low_high(df, v):.3f}" for v in balance_vars]

# Add N row
n_row = pd.DataFrame(
    {
        "Low": [Ns.get("Low", 0)],
        "Medium": [Ns.get("Medium", 0)],
        "High": [Ns.get("High", 0)],
        "SMD (High-Low)": [""],
    },
    index=["N"],
)

balance_table = pd.concat([n_row, balance_table])

balance_table

/tmp/ipykernel_3287395/1393347766.py:116: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  means = df.groupby("tutor_y_tercile")[balance_vars].mean().T
/tmp/ipykernel_3287395/1393347766.py:117: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sds = df.groupby("tutor_y_tercile")[balance_vars].std().T
/tmp/ipykernel_3287395/1393347766.py:118: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  Ns = df.groupby("tutor_y_tercile").size()

,Low,Medium,High,SMD (High-Low)
N,56,56,56,
participacion_x,0.106 (0.050),0.102 (0.060),0.134 (0.075),0.439
participacion_x_count,8.893 (2.890),9.429 (3.861),9.607 (3.681),0.216
edad,10.000 (0.000),10.000 (0.000),10.000 (0.000),nan
grado,7.268 (1.446),7.518 (1.561),7.536 (1.464),0.184
tratamiento,nan (nan),nan (nan),nan (nan),nan
total_sessions,94.911 (34.055),88.839 (31.083),103.732 (35.600),0.253
sessions_with_X,33.982 (24.158),29.732 (16.457),44.232 (29.546),0.380
sessions_with_X_preY,52.732 (60.763),38.089 (34.841),73.429 (80.032),0.291
sessions_with_Y_preY,0.000 (0.000),0.000 (0.000),0.000 (0.000),nan


In [ ]:
#sessions_with_Y_preY es cuantas tuvo con el tutor Y antes de la mínima clase Y que se midió

Entiendo lo que te “huele mal”, pero ojo: que sessions_with_Y_preY sea 0 para todos SÍ puede ser perfectamente correcto si tu fecha_y está definida como “la primera sesión con tutor Y”.

De hecho, bajo esa definición:

fecha_y(i) = min(fecha_clase) donde tutor == Y para estudiante i

entonces no puede existir ninguna sesión con Y con fecha_clase < fecha_y(i)
⇒ sessions_with_Y_preY tiene que ser 0 (salvo errores de datos/definición).

Y el nan en SMD sale porque la variable es constante (varianza 0). No es “bug”; es la matemática del SMD.

In [17]:
# 1) ¿Cuántos estudiantes sí tienen fecha_y?
print("Share fecha_y NaT:", df["fecha_y"].isna().mean())
print("N con fecha_y:", df["fecha_y"].notna().sum())

# 2) Para estudiantes con fecha_y, revisa su min fecha con Y y compárala
tmp = events_xy.copy()  # events a nivel sesión con id_tutor_x/id_tutor_y
tmp_y = tmp[tmp["id_tutor"].eq(tmp["id_tutor_y"])].copy()

check = (tmp_y.groupby("id_estudiante")["fecha_clase"].agg(["min","count"])
         .rename(columns={"min":"min_fecha_con_Y", "count":"n_sesiones_con_Y"}))

check = check.merge(df[["id_estudiante","fecha_y"]], on="id_estudiante", how="left")
check["diff_days"] = (check["fecha_y"] - check["min_fecha_con_Y"]).dt.days

print(check["diff_days"].value_counts(dropna=False).head(10))
print(check.sort_values("diff_days").head(10))

Share fecha_y NaT: 0.0
N con fecha_y: 168
diff_days
0    168
Name: count, dtype: int64
   id_estudiante min_fecha_con_Y  n_sesiones_con_Y    fecha_y  diff_days
0           1144      2023-10-13                18 2023-10-13          0
1           4702      2023-10-30                18 2023-10-30          0
2           6240      2023-02-20                22 2023-02-20          0
3           8318      2023-10-23                32 2023-10-23          0
4           8318      2023-10-23                32 2023-10-23          0
5           9155      2023-02-13                21 2023-02-13          0
6          11302      2024-01-22                13 2024-01-22          0
7          11705      2023-02-20               201 2023-02-20          0
8          11705      2023-02-20               201 2023-02-20          0
9          11705      2023-02-20               201 2023-02-20          0


In [31]:
t1 = tratamiento[~(tratamiento[["id_estudiante" , "id_grupo"]].duplicated())]

In [22]:
len(tratamiento)

81615

In [34]:
t1[["id_estudiante" , "id_grupo"]].dropna()

,id_estudiante,id_grupo
233,15987,8273.0
234,46335,7950.0
235,50345,8226.0
236,17932,7287.0
237,7876,8243.0
...,...,...
81603,111167,26067.0
81604,111169,26073.0
81605,42643,26045.0
81606,71015,26089.0


In [38]:
tratamiento.columns

Index(['Unnamed: 0', 'id_estudiante', 'detalles_tratamiento',
       'asignacion_tratamiento', 'id_estrato', 'vigente',
       'id_estudiante_ronda', 'finalizacion_tratamiento',
       'fecha_asignacion_tutor', 'fecha_TH_end', 'fecha_OP_end',
       'clases_asignadas', 'clases_dadas', 'clases_asistidas',
       'id_ninyos_inicio', 'id_ninyos_fin', 'id_tutor', 'id_grupo',
       'score_mate_final', 'score_mate_inicial', 'id_encuesta_inicial',
       'id_encuesta_final', 'historial', 'id_tarea', 'op_end_asignada',
       'th_end_asignada', 'id_op', 'id_tarea_op', 'excepcion',
       'detalles_operacion_tratamiento', 'id_tratamiento'],
      dtype='object')

In [44]:
df[["id_estudiante" ]].isin(t1[["id_estudiante"]]).sum()

id_estudiante    0
dtype: int64

In [53]:
t2 = t1[(t1.id_estudiante.isin(df.id_estudiante))][["id_estudiante" , "id_grupo"]]

In [54]:
t2

,id_estudiante,id_grupo
254,27410,7524.0
395,23226,6874.0
479,45403,7077.0
494,33178,6884.0
508,24585,7949.0
...,...,...
81095,65361,25561.0
81159,18300,25560.0
81367,52124,25855.0
81432,41400,25928.0


In [69]:
mask = list(zip(df["id_estudiante"], df["id_grupo_x"])).__iter__()
pairs = set(zip(t2["id_estudiante"], t2["id_grupo"]))

mask = [(a, b) in pairs for a, b in zip(df["id_estudiante"], df["id_grupo_x"])]

df_filtered = df[mask]

In [70]:
df_filtered

,id_estudiante,id_grupo_x,id_grupo_y,id_tutor_x,id_tutor_y,fecha_x,correcta,participacion_x,participacion_y,participacion_x_count,...,edad,genero,grado,tratamiento,instituto,campus,total_sessions,sessions_with_X,sessions_with_Y,genero_female
0,1144,10564.0,14098.0,74957,89015,2023-02-27,True,0.137177,0.074346,6,...,10,F,7,NaN,6644932568,NaN,130,21,18,1
1,4702,7066.0,15628.0,55425,90234,2022-08-22,True,0.129860,0.134454,14,...,10,M,10,NaN,6862439465,NaN,137,25,18,0
2,6240,8423.0,10982.0,57145,69670,2022-08-22,True,0.200286,0.051800,5,...,10,M,7,NaN,6645824543,NaN,117,20,22,0
3,8318,13852.0,15441.0,88550,90297,2023-08-28,True,0.108801,0.216710,8,...,10,F,10,NaN,6644930182,NaN,96,54,16,1
4,9155,7026.0,10916.0,55863,65176,2022-08-15,True,0.235968,0.201839,13,...,10,M,7,NaN,6641880583,NaN,115,25,21,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,8318,13852.0,15441.0,88550,90297,2023-08-28,True,0.108801,0.216710,8,...,10,F,10,NaN,6644930182,NaN,96,54,16,1
163,28037,14073.0,15442.0,88550,90297,2023-08-28,True,0.183278,0.272177,12,...,10,M,7,NaN,RICARDO FLORES MAGÓN,NaN,136,76,47,0
165,48085,13852.0,15441.0,88550,90297,2023-08-28,True,0.133915,0.154955,6,...,10,M,10,NaN,15EES0521Z,NaN,92,108,41,0
166,57029,14620.0,15438.0,88522,90297,2023-10-05,True,0.077402,0.291155,8,...,10,M,5,NaN,15PPR29870,NaN,53,56,14,0


In [77]:
mask = list(zip(df["id_estudiante"], df["id_grupo_y"])).__iter__()
pairs = set(zip(t2["id_estudiante"], t2["id_grupo"]))

mask = [(a, b) in pairs for a, b in zip(df["id_estudiante"], df["id_grupo_y"])]

df_filtered2 = df[mask]

In [82]:
df_filtered2

,id_estudiante,id_grupo_x,id_grupo_y,id_tutor_x,id_tutor_y,fecha_x,correcta,participacion_x,participacion_y,participacion_x_count,...,edad,genero,grado,tratamiento,instituto,campus,total_sessions,sessions_with_X,sessions_with_Y,genero_female
0,1144,10564.0,14098.0,74957,89015,2023-02-27,True,0.137177,0.074346,6,...,10,F,7,NaN,6644932568,NaN,130,21,18,1
1,4702,7066.0,15628.0,55425,90234,2022-08-22,True,0.129860,0.134454,14,...,10,M,10,NaN,6862439465,NaN,137,25,18,0
2,6240,8423.0,10982.0,57145,69670,2022-08-22,True,0.200286,0.051800,5,...,10,M,7,NaN,6645824543,NaN,117,20,22,0
4,9155,7026.0,10916.0,55863,65176,2022-08-15,True,0.235968,0.201839,13,...,10,M,7,NaN,6641880583,NaN,115,25,21,0
5,11302,12970.0,16741.0,64162,91674,2023-05-29,True,0.103982,0.050061,16,...,10,M,10,NaN,6188069923,NaN,70,33,13,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,65361,11699.0,13797.0,69842,87461,2023-03-06,True,0.211271,0.138787,10,...,10,F,7,NaN,15EPRO668Z,NaN,101,17,26,1
157,68686,11490.0,13570.0,58968,85721,2023-02-27,True,0.089042,0.110976,5,...,10,F,9,NaN,21DSTO112Z,NaN,57,25,32,1
159,72906,11919.0,13386.0,78685,87558,2023-02-27,True,0.130349,0.140462,7,...,10,M,9,NaN,21DST0030P,NaN,45,18,21,0
160,80117,12375.0,13686.0,70649,87542,2023-02-27,True,0.056670,0.033595,9,...,10,F,9,NaN,DST0047P,NaN,44,21,20,1


In [83]:
t1[t1["id_estudiante"] == grupo_x.get() & t1["id_grupo"]]

,Unnamed: 0,id_estudiante,detalles_tratamiento,asignacion_tratamiento,id_estrato,vigente,id_estudiante_ronda,finalizacion_tratamiento,fecha_asignacion_tutor,fecha_TH_end,...,id_encuesta_final,historial,id_tarea,op_end_asignada,th_end_asignada,id_op,id_tarea_op,excepcion,detalles_operacion_tratamiento,id_tratamiento
0,0,53973,{'emocional': 0},2022-08-08 14:04:33,1.0,0,1,2023-08-16 08:07:47,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,{'emocional': 0},1.0
1,1,54283,{'emocional': 1},2022-08-08 15:11:25,2.0,0,2,2023-02-21 19:53:16,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,{'emocional': 1},2.0
2,2,45667,{'emocional': 0},2022-08-08 15:50:56,3.0,0,3,2023-08-16 08:07:47,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,{'emocional': 0},1.0
3,3,55925,{'emocional': 1},2022-08-08 16:14:32,4.0,0,4,2023-02-21 19:53:16,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,{'emocional': 1},2.0
4,4,45511,{'emocional': 0},2022-08-08 16:17:46,2.0,0,5,2023-02-21 19:53:16,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,{'emocional': 0},1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81610,81610,110862,"{'version': 'otonyo2024', 'actividades': 'pre_...",2026-03-02 15:19:08,5.0,1,95509,NaN,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,27.0
81611,81611,110863,"{'version': 'otonyo2024', 'actividades': 'pre_...",2026-03-02 15:19:08,5.0,1,95510,NaN,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,27.0
81612,81612,111202,"{'version': 'otonyo2024', 'actividades': 'pre_...",2026-03-02 16:20:59,48.0,1,95511,NaN,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,27.0
81613,81613,104697,"{'version': 'otonyo2024', 'actividades': 'pre_...",2026-03-02 17:56:03,3.0,1,95512,NaN,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,27.0


In [88]:
di=tratamiento.set_index(["id_estudiante", "id_grupo"])["id_tratamiento"].to_dict()

In [95]:
df_filtered

,id_estudiante,id_grupo_x,id_grupo_y,id_tutor_x,id_tutor_y,fecha_x,correcta,participacion_x,participacion_y,participacion_x_count,...,genero,grado,tratamiento,instituto,campus,total_sessions,sessions_with_X,sessions_with_Y,genero_female,tratamiento_x
0,1144,10564.0,14098.0,74957,89015,2023-02-27,True,0.137177,0.074346,6,...,F,7,NaN,6644932568,NaN,130,21,18,1,NaN
1,4702,7066.0,15628.0,55425,90234,2022-08-22,True,0.129860,0.134454,14,...,M,10,NaN,6862439465,NaN,137,25,18,0,NaN
2,6240,8423.0,10982.0,57145,69670,2022-08-22,True,0.200286,0.051800,5,...,M,7,NaN,6645824543,NaN,117,20,22,0,NaN
3,8318,13852.0,15441.0,88550,90297,2023-08-28,True,0.108801,0.216710,8,...,F,10,NaN,6644930182,NaN,96,54,16,1,NaN
4,9155,7026.0,10916.0,55863,65176,2022-08-15,True,0.235968,0.201839,13,...,M,7,NaN,6641880583,NaN,115,25,21,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,8318,13852.0,15441.0,88550,90297,2023-08-28,True,0.108801,0.216710,8,...,F,10,NaN,6644930182,NaN,96,54,16,1,NaN
163,28037,14073.0,15442.0,88550,90297,2023-08-28,True,0.183278,0.272177,12,...,M,7,NaN,RICARDO FLORES MAGÓN,NaN,136,76,47,0,NaN
165,48085,13852.0,15441.0,88550,90297,2023-08-28,True,0.133915,0.154955,6,...,M,10,NaN,15EES0521Z,NaN,92,108,41,0,NaN
166,57029,14620.0,15438.0,88522,90297,2023-10-05,True,0.077402,0.291155,8,...,M,5,NaN,15PPR29870,NaN,53,56,14,0,NaN


In [97]:
df_filtered["tratamiento_x"]=df_filtered.apply(lambda x: di.get((x.id_estudiante, x.id_grupo_x),-1), axis=1)

/tmp/ipykernel_3287395/2471322657.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["tratamiento_x"]=df_filtered.apply(lambda x: di.get((x.id_estudiante, x.id_grupo_x),-1), axis=1)


In [98]:
df_filtered["tratamiento_y"]=df_filtered.apply(lambda x: di.get((x.id_estudiante, x.id_grupo_y),-1), axis=1)

/tmp/ipykernel_3287395/4033175917.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["tratamiento_y"]=df_filtered.apply(lambda x: di.get((x.id_estudiante, x.id_grupo_y),-1), axis=1)


In [80]:
grupo_x = dict(zip(df_filtered["id_estudiante"], df_filtered["id_grupo_x"]))

grupo_y = dict(zip(df_filtered2["id_estudiante"], df_filtered2["id_grupo_y"]))

In [76]:
grupo_x

{1144: 10564.0,
 4702: 7066.0,
 6240: 8423.0,
 8318: 13852.0,
 9155: 7026.0,
 11302: 12970.0,
 11705: 12276.0,
 11892: 8471.0,
 14267: 8427.0,
 14753: 8145.0,
 14831: 7062.0,
 15549: 7997.0,
 15947: 11289.0,
 17392: 6859.0,
 17665: 7548.0,
 17836: 7426.0,
 17849: 10385.0,
 17994: 11131.0,
 18004: 11233.0,
 18049: 7968.0,
 19082: 6898.0,
 20100: 9506.0,
 20447: 8071.0,
 20841: 10693.0,
 21250: 12388.0,
 21273: 11260.0,
 21453: 10250.0,
 21742: 6891.0,
 22496: 6826.0,
 23226: 10416.0,
 23455: 6874.0,
 23521: 6841.0,
 24255: 11042.0,
 24585: 7949.0,
 24766: 6780.0,
 25152: 10970.0,
 25644: 11367.0,
 25646: 8040.0,
 25791: 11294.0,
 26418: 7937.0,
 27410: 7524.0,
 27584: 7949.0,
 27919: 8181.0,
 28037: 14073.0,
 28425: 8564.0,
 28443: 7474.0,
 28469: 10916.0,
 28725: 11006.0,
 28883: 11009.0,
 29268: 6884.0,
 29423: 6829.0,
 29531: 6891.0,
 29790: 11313.0,
 30746: 11186.0,
 31041: 7603.0,
 32062: 7160.0,
 32411: 11272.0,
 32475: 11260.0,
 32770: 7129.0,
 33178: 6884.0,
 33190: 8014.0,
 333

In [81]:
grupo_y

{1144: 14098.0,
 4702: 15628.0,
 6240: 10982.0,
 9155: 10916.0,
 11302: 16741.0,
 11705: 13344.0,
 11892: 14489.0,
 14267: 11125.0,
 14753: 10841.0,
 14831: 11171.0,
 15549: 10928.0,
 15947: 13652.0,
 17665: 15554.0,
 17836: 14732.0,
 17849: 14532.0,
 17994: 13440.0,
 18004: 15051.0,
 18049: 14384.0,
 18300: 13374.0,
 19082: 11272.0,
 20100: 14328.0,
 20447: 12292.0,
 20841: 14482.0,
 21250: 14891.0,
 21273: 14482.0,
 21453: 13560.0,
 21742: 11079.0,
 22496: 10840.0,
 23226: 13598.0,
 23455: 10970.0,
 23521: 10667.0,
 24255: 14738.0,
 24585: 14739.0,
 24766: 11000.0,
 25152: 13594.0,
 25644: 13399.0,
 25646: 10852.0,
 25791: 14708.0,
 26418: 14732.0,
 27410: 13440.0,
 27584: 14078.0,
 27919: 14728.0,
 28037: 14073.0,
 28425: 14047.0,
 28443: 11385.0,
 28469: 14056.0,
 28725: 14830.0,
 28883: 14728.0,
 29268: 13257.0,
 29423: 10928.0,
 29531: 11218.0,
 29790: 14728.0,
 30746: 14896.0,
 31041: 14345.0,
 32062: 14078.0,
 32411: 13662.0,
 32475: 14427.0,
 32770: 10970.0,
 33178: 11259.0,
 